# Nhóm 5 — SegNet & FCN Detection (Kaggle)

Notebook nhận diện `/kaggle/input` và `/kaggle/working`, gọi chung lõi dự án. Bật GPU trong Notebook Settings. Synthetic mode chỉ dành cho smoke-test.

In [ ]:
from pathlib import Path
import os, sys, torch
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
PROJECT_ROOT = Path('/kaggle/working/BTL03')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError('Clone/copy repository BTL03 vào /kaggle/working/BTL03 trước khi chạy.')
sys.path.insert(0, str(PROJECT_ROOT))
os.environ['BTL03_PROJECT_ROOT'] = str(PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Kaggle inputs:', [p.name for p in INPUT_ROOT.iterdir()])
print('Seed=42')

## Gắn dataset và chạy
Attach datasets trong Kaggle Input, sau đó cấu hình đường dẫn trong YAML hoặc sửa hai biến dưới đây để trỏ tới mount chứa dữ liệu. Không thay đổi mã trong `src/` vì đường dẫn môi trường thuộc notebook adapter.

In [ ]:
from scripts.run_group5_others import Group5OthersRunner
from src.utils import seed_everything
seed_everything(42)
runner = Group5OthersRunner(project_root=PROJECT_ROOT, member_alias='CHIVY2005')
# In dataset đã attach; sửa path YAML/runner theo đúng cấu trúc thực tế của dataset.
for name in [p.name for p in INPUT_ROOT.iterdir() if p.is_dir()]: print(name)
# Tự nhận diện các thư mục legacy phổ biến mà không đoán schema NPY tùy ý.
for depth in INPUT_ROOT.rglob('depth'):
    labels = depth.parent/'labels'
    if labels.is_dir(): print('SegNet depth/labels:', depth, labels)
for vehicle_root in INPUT_ROOT.rglob('vehicles'):
    if (vehicle_root.parent/'non-vehicles').is_dir(): print('FCN vehicle root:', vehicle_root.parent)
# Sau khi kiểm tra nguồn dữ liệu, gán đúng các đường dẫn ở YAML.
USE_SYNTHETIC_SMOKE = False
results = runner.run(task='all', synthetic=USE_SYNTHETIC_SMOKE)
results

In [ ]:
archive = runner.package_artifacts(WORK_ROOT/'group5_segnet_fcn_artifacts.zip')
print('Artifact ZIP:', archive, 'bytes:', archive.stat().st_size)
from IPython.display import display, Image
for path in (PROJECT_ROOT/'outputs/figures/t0210_segnet_segmentation_preview.png', PROJECT_ROOT/'outputs/figures/t0211_fcn_heatmap_detection.png'):
    if path.exists(): display(Image(filename=str(path)))